In [ ]:

# ============================================================
# BANK QUEUE & CUSTOMER SERVICE SIMULATION
# COMPLETE FIXED VERSION 2.0
# Simulation and Modeling Lab Project
# Google Colab
# ============================================================

!pip -q install simpy ipywidgets pandas matplotlib numpy

import simpy
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from collections import defaultdict
from google.colab import files

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

PROJECT_NAME = "Bank Queue & Customer Service Simulation"

DEFAULT_DURATION = 480
DEFAULT_TELLERS = 3
DEFAULT_ARRIVAL = 3
DEFAULT_SERVICE = 5
DEFAULT_SEED = 42

# ============================================================
# 2. BANK SIMULATION ENGINE
# ============================================================

class BankSimulation:

    def __init__(
        self,
        duration=480,
        tellers=3,
        arrival=3,
        service=5,
        seed=42
    ):

        self.duration = duration
        self.teller_count = tellers
        self.arrival = arrival
        self.service = service

        self.rng = random.Random(seed)
        self.env = simpy.Environment()

        # Teller pool
        self.available_tellers = simpy.Store(
            self.env,
            capacity=tellers
        )

        for i in range(1, tellers + 1):
            self.available_tellers.put(i)

        # Statistics
        self.total_customers = 0
        self.waiting_count = 0

        self.records = []

        self.teller_busy_time = defaultdict(float)
        self.busy_tellers = set()

        self.queue_history = []
        self.busy_history = []

        self.max_queue = 0

    # --------------------------------------------------------
    # CUSTOMER PROCESS
    # --------------------------------------------------------

    def customer(self, customer_id):

        arrival_time = self.env.now

        self.waiting_count += 1

        self.max_queue = max(
            self.max_queue,
            self.waiting_count
        )

        # Get the next available teller (FIFO)
        teller_id = yield self.available_tellers.get()

        self.waiting_count -= 1

        service_start = self.env.now

        waiting_time = service_start - arrival_time

        # Generate random service time
        service_duration = self.rng.expovariate(
            1 / self.service
        )

        self.busy_tellers.add(teller_id)

        # Process customer service
        yield self.env.timeout(service_duration)

        departure_time = self.env.now

        self.busy_tellers.remove(teller_id)

        # Calculate busy time during bank operating hours
        actual_busy_time = max(
            0,
            min(departure_time, self.duration)
            - min(service_start, self.duration)
        )

        self.teller_busy_time[teller_id] += actual_busy_time

        # Save customer information
        self.records.append({

            "Customer ID": customer_id,
            "Teller": teller_id,
            "Arrival Time": arrival_time,
            "Service Start": service_start,
            "Service Time": service_duration,
            "Waiting Time": waiting_time,
            "Departure Time": departure_time,
            "Time in System": departure_time - arrival_time

        })

        # Release teller
        yield self.available_tellers.put(teller_id)

    # --------------------------------------------------------
    # CUSTOMER ARRIVAL PROCESS
    # --------------------------------------------------------

    def arrival_process(self):

        customer_id = 0

        while True:

            interval = self.rng.expovariate(
                1 / self.arrival
            )

            yield self.env.timeout(interval)

            # Stop new arrivals after bank closing
            if self.env.now >= self.duration:
                break

            customer_id += 1

            self.total_customers += 1

            self.env.process(
                self.customer(customer_id)
            )

    # --------------------------------------------------------
    # MONITOR PROCESS
    # IMPORTANT: FIXED INFINITE LOOP
    # --------------------------------------------------------

    def monitor(self):

        while self.env.now < self.duration:

            self.queue_history.append({

                "Time": self.env.now,
                "Queue Length": self.waiting_count

            })

            self.busy_history.append({

                "Time": self.env.now,
                "Busy Tellers": len(self.busy_tellers)

            })

            yield self.env.timeout(1)

    # --------------------------------------------------------
    # RUN SIMULATION
    # --------------------------------------------------------

    def run(self):

        self.env.process(self.arrival_process())

        self.env.process(self.monitor())

        # Run until bank closing time
        self.env.run(until=self.duration)

        # Complete remaining customer services
        # Monitor and arrival processes have already stopped
        self.env.run()

        # Sort customer records by arrival ID
        self.records.sort(
            key=lambda x: x["Customer ID"]
        )

        return self

    # --------------------------------------------------------
    # CALCULATE RESULTS
    # --------------------------------------------------------

    def get_results(self):

        df = pd.DataFrame(self.records)

        if df.empty:
            return None

        total_busy = sum(
            self.teller_busy_time.values()
        )

        utilization = (
            total_busy /
            (self.teller_count * self.duration)
        ) * 100

        return {

            "Total Customers": self.total_customers,

            "Served Customers": len(df),

            "Average Waiting Time":
                df["Waiting Time"].mean(),

            "Average Service Time":
                df["Service Time"].mean(),

            "Average Time in System":
                df["Time in System"].mean(),

            "Teller Utilization": utilization,

            "Maximum Queue Length": self.max_queue,

            "Customer Records": df

        }


# ============================================================
# 3. DASHBOARD HEADER
# ============================================================

display(HTML("""
<div style="
    background:linear-gradient(120deg,#101827,#17365d);
    padding:28px;
    border-radius:15px;
    color:white;
    text-align:center;
    margin-bottom:20px;
">
    <h1 style="color:#60a5fa;">
        🏦 BANK QUEUE SIMULATION
    </h1>

    <h3>Customer Service Management System</h3>

    <p>Simulation and Modeling Lab Project</p>

    <p>Python | SimPy | Pandas | Matplotlib</p>
</div>
"""))


# ============================================================
# 4. SIMULATION CONTROLS
# ============================================================

teller_input = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    description="Tellers:",
    style={"description_width":"120px"}
)

arrival_input = widgets.IntSlider(
    value=3,
    min=1,
    max=20,
    description="Arrival Interval:",
    style={"description_width":"120px"}
)

service_input = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    description="Service Time:",
    style={"description_width":"120px"}
)

duration_input = widgets.IntSlider(
    value=480,
    min=60,
    max=720,
    step=60,
    description="Duration:",
    style={"description_width":"120px"}
)

seed_input = widgets.IntText(
    value=42,
    description="Random Seed:",
    style={"description_width":"120px"}
)

start_button = widgets.Button(
    description="START SIMULATION",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="190px")
)

reset_button = widgets.Button(
    description="RESET",
    button_style="danger",
    icon="refresh",
    layout=widgets.Layout(width="120px")
)

download_button = widgets.Button(
    description="DOWNLOAD CSV",
    button_style="info",
    icon="download",
    layout=widgets.Layout(width="170px")
)

output = widgets.Output()

current_simulation = {
    "sim": None,
    "comparison": None
}


# ============================================================
# 5. KPI DASHBOARD
# ============================================================

def show_kpi(data):

    cards = [

        ("Total Customers",
         data["Total Customers"]),

        ("Served Customers",
         data["Served Customers"]),

        ("Average Waiting",
         f'{data["Average Waiting Time"]:.2f} min'),

        ("Average Service",
         f'{data["Average Service Time"]:.2f} min'),

        ("Teller Utilization",
         f'{data["Teller Utilization"]:.2f}%'),

        ("Maximum Queue",
         data["Maximum Queue Length"])

    ]

    html = """
    <div style="
        display:grid;
        grid-template-columns:repeat(3,minmax(0,1fr));
        gap:12px;
        margin:20px 0;
    ">
    """

    for title, value in cards:

        html += f"""
        <div style="
            background:#172338;
            padding:20px;
            border-radius:12px;
            text-align:center;
            border:1px solid #334155;
        ">

        <div style="color:#93c5fd;font-size:14px;">
            {title}
        </div>

        <div style="
            color:#34d399;
            font-size:25px;
            font-weight:bold;
            margin-top:10px;
        ">
            {value}
        </div>

        </div>
        """

    html += "</div>"

    display(HTML(html))


# ============================================================
# 6. PERFORMANCE GRAPHS
# ============================================================

def show_graphs(sim):

    df = pd.DataFrame(sim.records)

    fig, axes = plt.subplots(
        2, 2,
        figsize=(15, 9)
    )

    # Graph 1: Queue Length

    queue_df = pd.DataFrame(
        sim.queue_history
    )

    axes[0, 0].plot(
        queue_df["Time"],
        queue_df["Queue Length"]
    )

    axes[0, 0].set_title(
        "Customer Waiting Queue"
    )

    axes[0, 0].set_xlabel("Time (Minutes)")
    axes[0, 0].set_ylabel("Queue Length")
    axes[0, 0].grid(True)

    # Graph 2: Busy Tellers

    busy_df = pd.DataFrame(
        sim.busy_history
    )

    axes[0, 1].step(
        busy_df["Time"],
        busy_df["Busy Tellers"],
        where="post"
    )

    axes[0, 1].set_title(
        "Busy Teller Monitoring"
    )

    axes[0, 1].set_xlabel("Time (Minutes)")
    axes[0, 1].set_ylabel("Busy Tellers")
    axes[0, 1].grid(True)

    # Graph 3: Waiting Time Distribution

    axes[1, 0].hist(
        df["Waiting Time"],
        bins=20,
        edgecolor="black"
    )

    axes[1, 0].set_title(
        "Waiting Time Distribution"
    )

    axes[1, 0].set_xlabel("Waiting Time (Minutes)")
    axes[1, 0].set_ylabel("Customers")
    axes[1, 0].grid(True)

    # Graph 4: Teller Utilization

    teller_ids = list(
        range(1, sim.teller_count + 1)
    )

    utilization = [

        sim.teller_busy_time[i] /
        sim.duration * 100

        for i in teller_ids

    ]

    axes[1, 1].bar(
        [f"Teller {i}" for i in teller_ids],
        utilization
    )

    axes[1, 1].set_title(
        "Individual Teller Utilization"
    )

    axes[1, 1].set_ylabel("Utilization (%)")
    axes[1, 1].set_ylim(0, 100)
    axes[1, 1].grid(axis="y")

    plt.tight_layout()
    plt.show()


# ============================================================
# 7. TELLER COMPARISON
# ============================================================

def compare_tellers():

    comparison = []

    for count in [1, 2, 3, 4]:

        test = BankSimulation(

            duration=duration_input.value,

            tellers=count,

            arrival=arrival_input.value,

            service=service_input.value,

            seed=seed_input.value

        )

        test.run()

        data = test.get_results()

        if data:

            comparison.append({

                "Tellers": count,

                "Customers Served":
                    data["Served Customers"],

                "Average Waiting Time":
                    data["Average Waiting Time"],

                "Teller Utilization":
                    data["Teller Utilization"]

            })

    comparison_df = pd.DataFrame(comparison)

    print("\nTELLER COMPARISON TABLE")

    display(
        comparison_df.round(2)
    )

    fig, axes = plt.subplots(
        1, 2,
        figsize=(13, 5)
    )

    axes[0].bar(
        comparison_df["Tellers"].astype(str),
        comparison_df["Average Waiting Time"]
    )

    axes[0].set_title(
        "Average Waiting Time Comparison"
    )

    axes[0].set_xlabel("Number of Tellers")
    axes[0].set_ylabel("Waiting Time (Minutes)")
    axes[0].grid(axis="y")

    axes[1].bar(
        comparison_df["Tellers"].astype(str),
        comparison_df["Teller Utilization"]
    )

    axes[1].set_title(
        "Teller Utilization Comparison"
    )

    axes[1].set_xlabel("Number of Tellers")
    axes[1].set_ylabel("Utilization (%)")
    axes[1].set_ylim(0, 100)
    axes[1].grid(axis="y")

    plt.tight_layout()
    plt.show()

    return comparison_df


# ============================================================
# 8. START SIMULATION BUTTON
# ============================================================

def start_simulation(button):

    start_button.disabled = True

    try:

        with output:

            clear_output(wait=True)

            print("Simulation is running...")

            sim = BankSimulation(

                duration=duration_input.value,

                tellers=teller_input.value,

                arrival=arrival_input.value,

                service=service_input.value,

                seed=seed_input.value

            )

            sim.run()

            current_simulation["sim"] = sim

            data = sim.get_results()

            if data is None:

                print("No customers arrived.")
                return

            print("\nSIMULATION COMPLETED SUCCESSFULLY!")

            print("=" * 55)

            # Dashboard
            show_kpi(data)

            # Customer Records
            print("\nCUSTOMER RECORDS")

            display(
                data["Customer Records"]
                .round(2)
                .head(30)
            )

            # Performance Graphs
            print("\nPERFORMANCE GRAPHS")

            show_graphs(sim)

            # Teller Comparison
            print("\nTELLER COMPARISON")

            comparison_df = compare_tellers()

            current_simulation["comparison"] = comparison_df

            # Final Summary
            print("\nFINAL SIMULATION REPORT")

            print("-" * 45)

            for key, value in data.items():

                if key == "Customer Records":
                    continue

                if isinstance(value, (float, np.floating)):

                    print(f"{key}: {value:.2f}")

                else:

                    print(f"{key}: {value}")

            print("-" * 45)

            print("PROJECT SIMULATION COMPLETED!")

    except Exception as error:

        with output:

            print("SIMULATION ERROR:")
            print(str(error))

    finally:

        start_button.disabled = False


# ============================================================
# 9. RESET BUTTON
# ============================================================

def reset_simulation(button):

    teller_input.value = DEFAULT_TELLERS

    arrival_input.value = DEFAULT_ARRIVAL

    service_input.value = DEFAULT_SERVICE

    duration_input.value = DEFAULT_DURATION

    seed_input.value = DEFAULT_SEED

    current_simulation["sim"] = None
    current_simulation["comparison"] = None

    with output:

        clear_output()

        print("Simulation settings reset successfully!")


# ============================================================
# 10. DOWNLOAD CSV BUTTON
# ============================================================

def download_data(button):

    sim = current_simulation["sim"]

    if sim is None:

        with output:

            print("Please run simulation first.")

        return

    df = pd.DataFrame(sim.records)

    df.to_csv(
        "bank_customer_records.csv",
        index=False
    )

    comparison = current_simulation["comparison"]

    if comparison is not None:

        comparison.to_csv(
            "teller_comparison.csv",
            index=False
        )

    files.download(
        "bank_customer_records.csv"
    )

    if comparison is not None:

        files.download(
            "teller_comparison.csv"
        )


# ============================================================
# 11. CONNECT BUTTON EVENTS
# ============================================================

start_button.on_click(
    start_simulation
)

reset_button.on_click(
    reset_simulation
)

download_button.on_click(
    download_data
)


# ============================================================
# 12. DISPLAY CONTROL PANEL
# ============================================================

display(HTML("""
<h2 style="color:#2563eb;">
⚙️ Simulation Configuration
</h2>

<p>
Change the parameters and start your bank simulation.
</p>
"""))

display(

    widgets.VBox([

        teller_input,

        arrival_input,

        service_input,

        duration_input,

        seed_input,

        widgets.HBox([

            start_button,

            reset_button,

            download_button

        ]),

        output

    ])

)

print("\nBANK QUEUE SIMULATION SYSTEM READY!")

print("Configure parameters and click START SIMULATION.")



BANK QUEUE SIMULATION SYSTEM READY!
Configure parameters and click START SIMULATION.
